In [2]:
import pandas as pd
import json
import re

df = pd.read_excel("annotation_finished.xlsx")
print(f"Geladen: {len(df)} Beispiele")
print(f"Spalten: {list(df.columns)}")

# Label-Verteilung
print(f"\nLabel-Verteilung:")
print(df["label"].fillna("annotated").value_counts())

Geladen: 100 Beispiele
Spalten: ['question_id', 'clapnq_id', 'question', 'title', 'passage_text', 'title_span_1', 'span_1', 'span_2', 'span_3', 'span_4', 'span_5', 'span_6', 'label', 'notes']

Label-Verteilung:
label
annotated          87
unanswerable        9
ill-formed          2
unclear_passage     1
partial             1
Name: count, dtype: int64


In [4]:
!pip install thefuzz

In [5]:
from thefuzz import fuzz, process as fuzz_process

def find_span_offset(span_text, source_text):
    """Find exact character offsets of a span in the source text using fuzzy matching + validation."""
    span_text = span_text.strip()
    
    # Try exact match first
    idx = source_text.find(span_text)
    if idx != -1:
        return {"text": span_text, "start": idx, "end": idx + len(span_text), "match": "exact"}
    
    # Fuzzy: sliding window
    best_score = 0
    best_start = 0
    best_len = len(span_text)
    
    for window_len in range(len(span_text) - 10, len(span_text) + 10):
        if window_len <= 0 or window_len > len(source_text):
            continue
        for start in range(0, len(source_text) - window_len + 1):
            candidate = source_text[start:start + window_len]
            score = fuzz.ratio(span_text, candidate)
            if score > best_score:
                best_score = score
                best_start = start
                best_len = window_len
    
    matched_text = source_text[best_start:best_start + best_len]
    return {
        "text": span_text,
        "matched_text": matched_text,
        "start": best_start,
        "end": best_start + best_len,
        "match": "exact" if best_score == 100 else f"fuzzy_{best_score}",
        "score": best_score
    }

# Span-Spalten identifizieren
span_cols = [c for c in df.columns if re.match(r"^span_\d+$", c)]
title_span_cols = [c for c in df.columns if c.startswith("title_span")]

print(f"Span-Spalten: {span_cols}")
print(f"Title-Span-Spalten: {title_span_cols}")

Span-Spalten: ['span_1', 'span_2', 'span_3', 'span_4', 'span_5', 'span_6']
Title-Span-Spalten: ['title_span_1']


In [6]:
results = []
errors = []

for idx, row in df.iterrows():
    entry = {
        "question_id": row["question_id"],
        "clapnq_id": str(row["clapnq_id"]),
        "title": row["title"],
        "question": row["question"],
        "passage_text": row["passage_text"],
        "label": row.get("label", None) if pd.notna(row.get("label", None)) else None,
        "notes": row.get("notes", None) if pd.notna(row.get("notes", None)) else None,
        "title_spans": [],
        "passage_spans": [],
    }
    
    # Title spans
    for col in title_span_cols:
        if pd.notna(row[col]) and str(row[col]).strip():
            offset = find_span_offset(str(row[col]), str(row["title"]))
            entry["title_spans"].append(offset)
            if offset["match"] != "exact":
                errors.append((row["question_id"], col, offset))
    
    # Passage spans
    for col in span_cols:
        if pd.notna(row[col]) and str(row[col]).strip():
            offset = find_span_offset(str(row[col]), str(row["passage_text"]))
            entry["passage_spans"].append(offset)
            if offset["match"] != "exact":
                errors.append((row["question_id"], col, offset))
    
    results.append(entry)

print(f"Verarbeitet: {len(results)} Beispiele")
print(f"Exakte Matches: {len(results) - len(errors)} / {sum(len(r['passage_spans']) + len(r['title_spans']) for r in results)} Spans")
print(f"Nicht-exakte Matches: {len(errors)}")

Verarbeitet: 100 Beispiele
Exakte Matches: 100 / 193 Spans
Nicht-exakte Matches: 0


In [7]:
if errors:
    print("Nicht-exakte Matches (manuell prüfen!):\n")
    for qid, col, offset in errors:
        print(f"  {qid} | {col}")
        print(f"    Annotiert: {offset['text'][:80]}...")
        print(f"    Gefunden:  {offset['matched_text'][:80]}...")
        print(f"    Score: {offset['score']}")
        print()
else:
    print("Alle Spans exakt gefunden ✓")

Alle Spans exakt gefunden ✓


In [8]:
# Nur saubere Felder exportieren
export = []
for r in results:
    entry = {
        "question_id": r["question_id"],
        "clapnq_id": r["clapnq_id"],
        "title": r["title"],
        "question": r["question"],
        "label": r["label"],
        "title_spans": [{"text": s["text"], "start": s["start"], "end": s["end"]} for s in r["title_spans"]],
        "passage_spans": [{"text": s["text"], "start": s["start"], "end": s["end"]} for s in r["passage_spans"]],
        "notes": r["notes"],
    }
    export.append(entry)

with open("annotations_gold.json", "w", encoding="utf-8") as f:
    json.dump(export, f, indent=2, ensure_ascii=False)

print(f"Gespeichert: annotations_gold.json ({len(export)} Beispiele)")

# Quick stats
n_annotated = sum(1 for r in export if r["passage_spans"] or r["title_spans"])
n_skipped = sum(1 for r in export if r["label"] in ["unanswerable", "ill-formed", "unclear_passage"])
avg_spans = sum(len(r["passage_spans"]) for r in export if r["passage_spans"]) / max(n_annotated, 1)

print(f"Annotiert: {n_annotated}")
print(f"Übersprungen: {n_skipped}")
print(f"Durchschnittliche Spans pro Beispiel: {avg_spans:.1f}")

Gespeichert: annotations_gold.json (100 Beispiele)
Annotiert: 87
Übersprungen: 12
Durchschnittliche Spans pro Beispiel: 2.2


In [9]:
print(df.iloc[0]["title"])
print(df.iloc[0]["question"])

Quality of life
who is quality of human being is necessary to lead a happy life
